# Epic 7 -- Multi-Spectral Rendering Pipeline

This notebook demonstrates the full rendering pipeline: from material layout
to multi-spectral image to RGB visualisation and anomaly scoring.

## Pipeline

```
Material Layout  -->  Spectral Renderer  -->  [C, H, W] Image
  + Defects                                      |
                                                  v
                                         RGB Visualisation
                                         Anomaly Scoring
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from udm_epic7.spectral.wavelength_model import default_spectral_config
from udm_epic7.rendering.spectral_renderer import (
    render_spectral_image,
    spectral_to_rgb,
)
from udm_epic7.evaluation.spectral_metrics import spectral_anomaly_score_from_material

In [ ]:
cfg = default_spectral_config()

# Create a material layout: mold_compound (0), silicon (1), copper (2)
layout = np.zeros((256, 256), dtype=np.int32)
layout[64:192, 64:192] = 1   # silicon die
layout[40:60, 80:176] = 2    # copper trace top
layout[196:216, 80:176] = 2  # copper trace bottom

# Render without defects
img_clean = render_spectral_image(layout, config=cfg, height=256, width=256,
                                  rng=np.random.default_rng(42))
print(f"Clean image shape: {img_clean.shape}")

In [ ]:
# Render with an oxidation defect on the copper trace
defects = [{
    'type': 'oxidation',
    'bbox': (40, 80, 60, 176),
    'severity': 0.7,
    'material': 'copper',
}]
img_defect = render_spectral_image(layout, defects=defects, config=cfg,
                                   height=256, width=256,
                                   rng=np.random.default_rng(42))

# Visualise
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: individual channels
for i, (ax, wl) in enumerate(zip(axes[0], [0, 1, 3])):
    ax.imshow(img_defect[wl], cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f'{cfg.wavelengths[wl]} nm')
    ax.axis('off')

# Row 2: RGB + anomaly
rgb_clean = spectral_to_rgb(img_clean, cfg)
rgb_defect = spectral_to_rgb(img_defect, cfg)
anomaly = spectral_anomaly_score_from_material(img_defect, 'copper', cfg)

axes[1][0].imshow(rgb_clean)
axes[1][0].set_title('Clean RGB')
axes[1][0].axis('off')

axes[1][1].imshow(rgb_defect)
axes[1][1].set_title('Defective RGB')
axes[1][1].axis('off')

im = axes[1][2].imshow(anomaly, cmap='hot')
axes[1][2].set_title('Anomaly Score vs Copper')
axes[1][2].axis('off')
fig.colorbar(im, ax=axes[1][2], fraction=0.046)

fig.suptitle('Multi-Spectral Rendering Pipeline', fontsize=15)
fig.tight_layout()
plt.show()

**Observations:**

- The oxidation defect is clearly visible in the 850 nm (near-IR) channel where
  copper has its highest reflectance contrast.
- The anomaly score map highlights the defective region automatically.
- RGB visualisation gives a reasonable false-colour representation for human review.